### The Transformer Architecture

The original **Transformer** (Vaswani et al., 2017) is an **encoder-decoder** architecture designed for sequence-to-sequence tasks like machine translation.

While GPT uses only the **decoder** (with causal masking), the full Transformer has two main parts:

1. **Encoder** — reads the entire input sequence bidirectionally (no masking)
2. **Decoder** — generates output tokens one at a time, attending to both previous outputs and the encoder's representations

Key components we will build step by step:
- **Sinusoidal positional encoding** (fixed, not learned)
- **Multi-head self-attention** (already covered, reused here)
- **Cross-attention** (decoder attends to encoder outputs)
- **Feed-forward network**
- **Add & Norm** (residual connection + layer normalization)
- **Full Encoder** and **Full Decoder** blocks

In [1]:
import torch
import torch.nn as nn
import math

### 1. Sinusoidal Positional Encoding

Unlike GPT which **learns** positional embeddings, the original Transformer uses **fixed sinusoidal** functions.

For each position `pos` and dimension `i`:
- PE(pos, 2i) = sin(pos / 10000^(2i / d_model))
- PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

This lets the model easily learn relative positions because PE(pos+k) can be represented as a linear function of PE(pos).

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

### 2. Multi-Head Attention (reused)

This is the same multi-head attention mechanism covered in notebook 7.
We add an **optional mask** argument so the same class works for:
- **Encoder self-attention**: no mask (bidirectional)
- **Decoder self-attention**: causal mask (look left only)
- **Decoder cross-attention**: no mask, but queries come from decoder, keys/values from encoder

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Linear projections + split into heads
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)

        # Concatenate heads
        context = context.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model)
        return self.out_proj(context)

### 3. Feed-Forward Network

Each Transformer block contains a position-wise FFN:
FFN(x) = Linear(d_model → d_ff) → ReLU → Linear(d_ff → d_model)

Traditionally `d_ff = 4 * d_model`.

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

### 4. Transformer Encoder Block

The encoder processes the input sequence **bidirectionally** — every token can attend to every other token.

Structure:
- Self-attention (no mask)
- Add & Norm (residual + layer norm)
- Feed-forward
- Add & Norm

In [5]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + Add & Norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward + Add & Norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

### 5. Transformer Decoder Block

The decoder has **two attention sub-layers**:
1. **Masked self-attention** — each token attends only to past tokens (causal)
2. **Cross-attention** — queries come from the decoder, keys/values come from the **encoder output**

This cross-attention is what makes the encoder-decoder architecture powerful — the decoder looks at the input while generating output.

In [6]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Masked self-attention + Add & Norm
        attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Cross-attention (queries from decoder, keys/values from encoder) + Add & Norm
        cross_out = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_out))

        # Feed-forward + Add & Norm
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

### 6. Full Transformer (Encoder + Decoder)

Putting it all together:
- Input tokens → Embedding + Positional encoding → **N Encoder blocks**
- Output tokens → Embedding + Positional encoding → **N Decoder blocks** (with encoder output)
- Decoder output → Linear → Softmax → vocabulary probabilities

We also include padding masks so the model ignores "&lt;pad&gt;" tokens.

In [7]:
class Transformer(nn.Module):
    def __init__(self,
                 src_vocab_size,
                 tgt_vocab_size,
                 d_model=512,
                 n_heads=8,
                 n_layers=6,
                 d_ff=2048,
                 max_len=100,
                 dropout=0.1):
        super().__init__()

        self.encoder_embed = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def make_padding_mask(self, seq, pad_idx=0):
        # mask shape: (batch, 1, 1, seq_len) for broadcasting over heads
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

    def make_causal_mask(self, seq):
        sz = seq.size(1)
        mask = torch.triu(torch.ones(sz, sz, device=seq.device), diagonal=1).bool()
        return ~mask.unsqueeze(0).unsqueeze(1)  # (1, 1, sz, sz)

    def forward(self, src, tgt):
        # src, tgt shapes: (batch, seq_len)
        src_mask = self.make_padding_mask(src)
        tgt_pad_mask = self.make_padding_mask(tgt)
        tgt_causal_mask = self.make_causal_mask(tgt)
        tgt_mask = tgt_pad_mask & tgt_causal_mask  # combine padding + causal

        # Encode
        src_emb = self.pos_enc(self.encoder_embed(src))
        for layer in self.encoder_layers:
            src_emb = layer(src_emb, src_mask)

        # Decode
        tgt_emb = self.pos_enc(self.decoder_embed(tgt))
        for layer in self.decoder_layers:
            tgt_emb = layer(tgt_emb, src_emb, src_mask, tgt_mask)

        # Output projection
        return self.fc_out(tgt_emb)

### 7. Demonstration on a Toy Task

Let's test the Transformer on a simple **copy task**: learn to output the same sequence as input.

This validates that:
- The encoder correctly processes the source
- The decoder correctly uses cross-attention to copy from encoder
- The causal mask prevents the decoder from cheating by looking ahead

In [8]:
def generate_copy_batch(batch_size, seq_len, vocab_size):
    # Random sequences of integers
    data = torch.randint(2, vocab_size, (batch_size, seq_len))  # skip 0 (pad) and 1 (start)
    src = data
    # decoder input: prepend a start token (1) and drop last token
    tgt_input = torch.cat([torch.ones(batch_size, 1, dtype=torch.long), data[:, :-1]], dim=1)
    # target: the original sequence (for loss computation)
    tgt_output = data
    return src, tgt_input, tgt_output

vocab_size = 20
batch_size = 32
seq_len = 10

src, tgt_in, tgt_out = generate_copy_batch(batch_size, seq_len, vocab_size)
print("src shape:", src.shape)
print("tgt_in shape:", tgt_in.shape)
print("tgt_out shape:", tgt_out.shape)
print("\nFirst sample:")
print("src:", src[0].tolist())
print("tgt_in:", tgt_in[0].tolist())
print("tgt_out:", tgt_out[0].tolist())

src shape: torch.Size([32, 10])
tgt_in shape: torch.Size([32, 10])
tgt_out shape: torch.Size([32, 10])

First sample:
src: [14, 4, 15, 12, 9, 9, 17, 11, 15, 10]
tgt_in: [1, 14, 4, 15, 12, 9, 9, 17, 11, 15]
tgt_out: [14, 4, 15, 12, 9, 9, 17, 11, 15, 10]


In [9]:
torch.manual_seed(42)

model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=64,
    n_heads=4,
    n_layers=3,
    d_ff=256,
    max_len=seq_len + 5,
    dropout=0.1
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore padding

def train_step(src, tgt_in, tgt_out):
    model.train()
    optimizer.zero_grad()
    logits = model(src, tgt_in)  # (batch, seq_len, vocab_size)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, num_batches=10):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for _ in range(num_batches):
            src, tgt_in, tgt_out = generate_copy_batch(32, seq_len, vocab_size)
            logits = model(src, tgt_in)
            preds = logits.argmax(dim=-1)
            mask = tgt_out != 0
            correct += (preds == tgt_out)[mask].sum().item()
            total += mask.sum().item()
    return correct / total

print("Training on copy task...")
for epoch in range(20):
    total_loss = 0.0
    for _ in range(50):
        src, tgt_in, tgt_out = generate_copy_batch(batch_size, seq_len, vocab_size)
        total_loss += train_step(src, tgt_in, tgt_out)
    avg_loss = total_loss / 50
    acc = evaluate(model)
    print(f"Epoch {epoch+1:2d} | loss: {avg_loss:.4f} | accuracy: {acc:.3f}")

Training on copy task...
Epoch  1 | loss: 2.7124 | accuracy: 0.218
Epoch  2 | loss: 1.9853 | accuracy: 0.665
Epoch  3 | loss: 1.1131 | accuracy: 0.914
Epoch  4 | loss: 0.5992 | accuracy: 0.976
Epoch  5 | loss: 0.4013 | accuracy: 0.989
Epoch  6 | loss: 0.3064 | accuracy: 0.993
Epoch  7 | loss: 0.2568 | accuracy: 0.988
Epoch  8 | loss: 0.2300 | accuracy: 0.998
Epoch  9 | loss: 0.1877 | accuracy: 0.996
Epoch 10 | loss: 0.1747 | accuracy: 0.993
Epoch 11 | loss: 0.1739 | accuracy: 0.999
Epoch 12 | loss: 0.1491 | accuracy: 0.998
Epoch 13 | loss: 0.1435 | accuracy: 0.997
Epoch 14 | loss: 0.1339 | accuracy: 0.989
Epoch 15 | loss: 0.1377 | accuracy: 0.999
Epoch 16 | loss: 0.1192 | accuracy: 0.997
Epoch 17 | loss: 0.1082 | accuracy: 0.998
Epoch 18 | loss: 0.1110 | accuracy: 0.994
Epoch 19 | loss: 0.1171 | accuracy: 0.992
Epoch 20 | loss: 0.1047 | accuracy: 1.000


In [10]:
def greedy_decode(model, src, max_len, start_token=1, pad_idx=0):
    model.eval()
    batch_size = src.size(0)
    tgt = torch.ones(batch_size, 1, dtype=torch.long) * start_token

    with torch.no_grad():
        for _ in range(max_len):
            logits = model(src, tgt)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tgt = torch.cat([tgt, next_token], dim=1)
    return tgt

src, _, _ = generate_copy_batch(3, seq_len, vocab_size)
output = greedy_decode(model, src, max_len=seq_len + 2)

for i in range(3):
    print(f"Sample {i+1}:")
    print(f"  Input:  {src[i].tolist()}")
    print(f"  Output: {output[i, 1:].tolist()}")  # skip start token

Sample 1:
  Input:  [15, 13, 11, 2, 7, 18, 11, 13, 2, 11]
  Output: [15, 13, 11, 2, 7, 18, 11, 13, 2, 11, 11, 11]
Sample 2:
  Input:  [19, 2, 11, 3, 3, 10, 17, 6, 7, 7]
  Output: [19, 2, 11, 3, 3, 10, 17, 6, 7, 7, 7, 7]
Sample 3:
  Input:  [7, 6, 3, 7, 17, 17, 3, 13, 13, 15]
  Output: [7, 6, 3, 7, 17, 17, 3, 13, 13, 15, 15, 3]


### Summary

The **Encoder-Decoder Transformer** adds two key ideas beyond the decoder-only GPT:

1. **Bidirectional encoder** — the encoder reads all input tokens at once, useful for understanding tasks (translation, summarization)
2. **Cross-attention** — the decoder attends to the encoder's output at every step, connecting input and output

Together these form the architecture described in "Attention Is All You Need", which powers models like:
- **T5** (encoder-decoder)
- **BART** (encoder-decoder)
- **Bidirectional encoders** (BERT) use the encoder alone
- **Autoregressive decoders** (GPT) use the decoder alone